In [ ]:
import pandas as pd
import numpy as np
import pyterrier_deepct
import pyterrier as pt
if not pt.started():
  pt.init(boot_packages=["com.github.terrierteam:terrier-prf:-SNAPSHOT"])

In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics2020.xml')
data = tree.getroot()
titles = []
title_ids = []
nums = []
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics.xml')
data = tree.getroot()    
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
topics = pd.DataFrame()
topics['qid'] = nums
topics['query'] = titles

qrels_path=("/workspace/src/evaluate_targers/qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

docnos = qrels.docno

In [ ]:
import os
rootdir = "/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/DeepCT_indexes/"

for subdir, dirs, files in os.walk(rootdir):
    for file in files:
        if(file == 'data.properties'):
            print(subdir)
            path = subdir + '/'+ file
            indexref = pt.IndexRef.of(path)
            index = pt.IndexFactory.of(indexref)
            dph = pt.BatchRetrieve(index, wmodel="DPH")
            dph_rm3 = (pt.BatchRetrieve(indexref, wmodel="DPH") >> 
            pt.rewrite.RM3(indexref) >> 
            pt.BatchRetrieve(indexref, wmodel="DPH"))
            bm25_rm3 = (pt.BatchRetrieve(indexref, wmodel="BM25") >> 
            pt.rewrite.RM3(indexref) >> 
            pt.BatchRetrieve(indexref, wmodel="BM25"))
            tf_idf = pt.BatchRetrieve(index, wmodel="TF_IDF")
            bm25 = pt.BatchRetrieve(index, wmodel="BM25")
            dlm = pt.BatchRetrieve(index, wmodel="DirichletLM")
            dlm_rm3 = (pt.BatchRetrieve(indexref, wmodel="DirichletLM") >> 
            pt.rewrite.RM3(indexref) >> 
            pt.BatchRetrieve(indexref, wmodel="DirichletLM"))
            models = [dph, dph_rm3, tf_idf, bm25_rm3, bm25, dlm, dlm_rm3]
            print(pt.Experiment(models, topics, qrels, eval_metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"]))
            

In [ ]:
import os
rootdir = "/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/DeepCT_indexes/"

for subdir, dirs, files in os.walk(rootdir):
    for file in files:
        if(file == 'data.properties'):
            print('\n' + subdir)
            path = subdir + '/'+ file
            indexref = pt.IndexRef.of(path)
            index = pt.IndexFactory.of(indexref)
            dph = pt.BatchRetrieve(index, wmodel="DPH")
            dph_rm3 = (pt.BatchRetrieve(indexref, wmodel="DPH") >> 
            pt.rewrite.RM3(indexref) >> 
            pt.BatchRetrieve(indexref, wmodel="DPH"))
            bm25_rm3 = (pt.BatchRetrieve(indexref, wmodel="BM25") >> 
            pt.rewrite.RM3(indexref) >> 
            pt.BatchRetrieve(indexref, wmodel="BM25"))
            tf_idf = pt.BatchRetrieve(index, wmodel="TF_IDF")
            bm25 = pt.BatchRetrieve(index, wmodel="BM25")
            dlm = pt.BatchRetrieve(index, wmodel="DirichletLM")
            dlm_rm3 = (pt.BatchRetrieve(indexref, wmodel="DirichletLM") >> 
            pt.rewrite.RM3(indexref) >> 
            pt.BatchRetrieve(indexref, wmodel="DirichletLM"))
            models = [dph, dph_rm3, tf_idf, bm25_rm3, bm25, dlm, dlm_rm3]
        
            for model in models:
                res= model.transform(topics)
                res_copy = res[res['docno'].isin(docnos)].copy()
                print(f'model : {model}, res: {pt.Utils.evaluate(res_copy,qrels,metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"])}')
            